# Diffusion from scratch: a DDPM on two moons (optional)
**LLMs, Animated · Lesson G1.3**

You'll build the complete DDPM recipe (Ho et al., 2020) on a 2-D toy dataset, the same two moons as in the lesson's animation:

1. the linear noise schedule and $\bar\alpha_t$,
2. one-jump forward noising $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$,
3. a sinusoidal time embedding for a small MLP noise predictor,
4. the training loss $\|\varepsilon - \varepsilon_\theta(x_t, t)\|^2$,
5. the 1,000-step sampling loop.

Each exercise has a `# TODO` cell followed by a **check** cell with `assert`s: run it, and silence (or a ✓) means success. Training takes well under a minute on a CPU; PyTorch is pre-installed on Colab. Solutions are at the very bottom.

In [ ]:
import math, time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)
torch.set_num_threads(2)   # small models are fastest on a couple of threads

def make_moons(n, noise=0.08, seed=None):
    # two interleaved half-circles, centred and scaled like the lesson's toy
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    th = torch.rand(n, generator=g) * math.pi
    up = torch.rand(n, generator=g) < 0.5
    x = torch.where(up, torch.cos(th), 1 - torch.cos(th))
    y = torch.where(up, torch.sin(th), 0.5 - torch.sin(th))
    pts = (torch.stack([x, y], 1) - torch.tensor([0.5, 0.25])) * 1.3
    return pts + noise * torch.randn(n, 2, generator=g)

data = make_moons(2000, seed=0)
plt.figure(figsize=(4, 3)); plt.scatter(data[:, 0], data[:, 1], s=2); plt.axis("equal"); plt.title("training data"); plt.show()

## 1 · The noise schedule
Use $T = 1000$ steps with $\beta_t$ rising **linearly** from $10^{-4}$ to $0.02$. Then $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s \le t} \alpha_s$ (a cumulative product).

Store them as 1-D tensors of length `T`, indexed `0 … T-1` (index `t` holds step `t + 1` of the paper; that off-by-one is the usual code convention).

In [ ]:
T = 1000
# TODO: betas, alphas, alpha_bar (each a float tensor of shape (T,))
betas = None
alphas = None
alpha_bar = None

**Check 1.** The lesson's numbers: $\bar\alpha_{250} \approx 0.524$ and $\bar\alpha_{1000} \approx 4.0 \times 10^{-5}$.

In [ ]:
assert betas.shape == (T,) and alpha_bar.shape == (T,)
assert abs(betas[0].item() - 1e-4) < 1e-9 and abs(betas[-1].item() - 0.02) < 1e-7
assert abs(alpha_bar[249].item() - 0.524) < 0.001, alpha_bar[249]
assert abs(alpha_bar[-1].item() - 4.04e-5) < 2e-6, alpha_bar[-1]
print("✓ schedule OK:  alpha_bar at t=250:", round(alpha_bar[249].item(), 4), " at t=1000:", f"{alpha_bar[-1].item():.2e}")

## 2 · Forward noising in one jump
Write `q_sample(x0, t, eps)` returning $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$.
`x0` and `eps` have shape `(B, 2)`; `t` is an integer tensor of shape `(B,)` with one step per row. (Hint: index `alpha_bar[t]`, then add a dimension with `[:, None]` so it broadcasts over the 2 coordinates.)

In [ ]:
def q_sample(x0, t, eps):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2.** The lesson's worked example ($x_0 = 1$, $t = 250$, $\varepsilon = -0.5 \Rightarrow x_t \approx 0.379$), and a look at the noised data.

In [ ]:
x = q_sample(torch.tensor([[1.0, 1.0]]), torch.tensor([249]), torch.tensor([[-0.5, -0.5]]))
assert abs(x[0, 0].item() - 0.379) < 0.002, x
xT = q_sample(data, torch.full((len(data),), T - 1), torch.randn_like(data))
assert abs(xT.std().item() - 1) < 0.08, "at t = T the data should look like N(0, I)"
fig, axs = plt.subplots(1, 4, figsize=(13, 3))
for ax, t in zip(axs, [0, 99, 249, 999]):
    xt = q_sample(data, torch.full((len(data),), t), torch.randn_like(data))
    ax.scatter(xt[:, 0], xt[:, 1], s=2); ax.set_title(f"t = {t + 1}"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.show(); print("✓ forward process OK")

## 3 · A time embedding
The network must know the noise level. Encode step `t` the transformer way (M1.3): for `half = dim // 2` frequencies $f_k = 10000^{-k/\text{half}}$, $k = 0 \dots \text{half}-1$, return

$$[\sin(t f_0), \dots, \sin(t f_{\text{half}-1}),\; \cos(t f_0), \dots, \cos(t f_{\text{half}-1})]$$

as a `(B, dim)` tensor.

In [ ]:
def timestep_embedding(t, dim=32):
    # TODO: t is an integer tensor of shape (B,)
    raise NotImplementedError("Exercise 3")

In [ ]:
e = timestep_embedding(torch.tensor([0, 10, 999]), 32)
assert e.shape == (3, 32)
assert torch.allclose(e[0, :16], torch.zeros(16)) and torch.allclose(e[0, 16:], torch.ones(16)), "t = 0: sines 0, cosines 1"
assert abs(e[1, 0].item() - math.sin(10)) < 1e-5
print("✓ embedding OK")

The noise predictor is a small MLP that sees the 2-D point and the embedding of `t` (provided):

In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self, hidden=128, tdim=32):
        super().__init__()
        self.tdim = tdim
        self.net = nn.Sequential(
            nn.Linear(2 + tdim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, timestep_embedding(t, self.tdim)], dim=1))

model = NoisePredictor()
print(sum(p.numel() for p in model.parameters()), "parameters")

## 4 · The training loss
Write `ddpm_loss(model, x0, t, eps)`: build $x_t$ with `q_sample`, predict $\hat\varepsilon = $ `model(x_t, t)`, and return the mean squared error to `eps` (a scalar). The training loop below draws `t` uniformly and `eps` from $\mathcal{N}(0, I)$.

In [ ]:
def ddpm_loss(model, x0, t, eps):
    # TODO
    raise NotImplementedError("Exercise 4")

In [ ]:
zero_model = lambda x, t: torch.zeros_like(x)
x0 = make_moons(64, seed=5); t = torch.randint(0, T, (64,)); eps = torch.randn(64, 2)
l = ddpm_loss(zero_model, x0, t, eps)
assert l.dim() == 0 and torch.allclose(l, (eps ** 2).mean()), "a model predicting 0 should score mean(eps²)"
print("✓ loss OK")

Now train (about 15–30 seconds on a CPU). The loss can't reach 0, since even a perfect model can't know the exact noise at high $t$, but it should drop clearly.

In [ ]:
torch.manual_seed(0)
model = NoisePredictor()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses, t0 = [], time.time()
for step in range(3000):
    x0 = make_moons(512)
    t = torch.randint(0, T, (512,))
    loss = ddpm_loss(model, x0, t, torch.randn_like(x0))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
print(f"trained in {time.time() - t0:.1f}s; final loss (avg of last 200) {sum(losses[-200:]) / 200:.3f}")
plt.figure(figsize=(5, 2.5)); plt.plot(losses, lw=0.5); plt.ylim(0, 1.2); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

## 5 · Sampling
Implement DDPM ancestral sampling. Start from $x \sim \mathcal{N}(0, I)$ and for `t = T-1, …, 0`:

$$x \leftarrow \frac{1}{\sqrt{\alpha_t}}\left(x - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x, t)\right) + \sqrt{\beta_t}\, z, \qquad z \sim \mathcal{N}(0, I),$$

with **no** noise on the last step (`t == 0`). Wrap it in `torch.no_grad()`.

In [ ]:
def sample(model, n):
    # TODO: return a (n, 2) tensor of samples
    raise NotImplementedError("Exercise 5")

**Check 5.** Samples should sit on the moons: we measure the average distance from each sample to its nearest neighbour in a large reference set.

In [ ]:
torch.manual_seed(1)
samples = sample(model, 1000)
ref = make_moons(5000, seed=1)
def mean_nn_dist(x):
    return torch.cdist(x, ref).min(dim=1).values.mean().item()
d = mean_nn_dist(samples)
print(f"avg distance to data: {d:.3f}  (pure noise would be about {mean_nn_dist(torch.randn(1000, 2)):.2f})")
assert samples.shape == (1000, 2)
assert d < 0.15, "samples should land on the moons"
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].scatter(ref[:1000, 0], ref[:1000, 1], s=2); axs[0].set_title("data")
axs[1].scatter(samples[:, 0], samples[:, 1], s=2, c="C1"); axs[1].set_title("DDPM samples (1,000 steps)")
for ax in axs: ax.set_xlim(-3, 3); ax.set_ylim(-2.2, 2.2)
plt.show(); print("✓ sampling OK")

## Experiment: DDIM with fewer steps
DDIM (Song, Meng & Ermon, 2020) reuses the **same** trained model but jumps deterministically between a few steps: predict $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon)/\sqrt{\bar\alpha_t}$, then $x_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t'}}\,\hat\varepsilon$.

Run the cell, then try `n_steps` of 2, 5, 10 and 50, and compare the distance to data and the pictures with the 1,000-step DDPM result. Other things to try: train for fewer steps, change `noise=` in `make_moons`, or swap the linear schedule for a cosine one.

In [ ]:
def ddim_sample(model, n, n_steps):
    ts = torch.linspace(T - 1, 0, n_steps).round().long().tolist()
    with torch.no_grad():
        x = torch.randn(n, 2)
        for i, t in enumerate(ts):
            eps = model(x, torch.full((n,), t))
            x0_hat = (x - (1 - alpha_bar[t]).sqrt() * eps) / alpha_bar[t].sqrt()
            ab_next = alpha_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)
            x = ab_next.sqrt() * x0_hat + (1 - ab_next).sqrt() * eps
    return x

torch.manual_seed(2)
fig, axs = plt.subplots(1, 4, figsize=(13, 3))
for ax, k in zip(axs, [2, 5, 10, 50]):
    s = ddim_sample(model, 1000, k)
    ax.scatter(s[:, 0], s[:, 1], s=2, c="C1"); ax.set_title(f"DDIM {k} steps: dist {mean_nn_dist(s):.3f}")
    ax.set_xlim(-3, 3); ax.set_ylim(-2.2, 2.2)
plt.show()

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def q_sample(x0, t, eps):
    ab = alpha_bar[t][:, None]
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def timestep_embedding(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def ddpm_loss(model, x0, t, eps):
    xt = q_sample(x0, t, eps)
    return ((model(xt, t) - eps) ** 2).mean()

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def sample(model, n):
    with torch.no_grad():
        x = torch.randn(n, 2)
        for t in range(T - 1, -1, -1):
            tt = torch.full((n,), t)
            eps = model(x, tt)
            x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
            if t > 0:
                x = x + betas[t].sqrt() * torch.randn_like(x)
    return x